# Module 49 — Geo-Experimentation: Difference-in-Differences (DiD)

> **Track 4 · Marketing Analytics & Optimization**  
> **Difficulty**: Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: Statsmodels, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 6 (Statistical Tests)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Geo-Experiment Data](#3-synthetic-geo-experiment-data)
4. [DiD Estimation](#4-did-estimation)
5. [Parallel Trends Validation](#5-parallel-trends-validation)
6. [Treatment Effect](#6-treatment-effect)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why geo-experimentation?

Geo-experiments **measure marketing impact** in real-world settings:
- **Causal inference**: Establish true marketing lift.
- **No user-level tracking**: Works with aggregate data.
- **Real-world conditions**: Tests in actual market conditions.

**Applications**:
1. **TV advertising**: Measure impact of TV campaigns.
2. **Regional campaigns**: Test campaigns in select regions.
3. **Pricing experiments**: Test price changes by region.

**Key concepts**:
- **Treatment region**: Region where campaign runs.
- **Control region**: Similar region without campaign.
- **Difference-in-Differences**: Compare changes over time.

In this notebook we will:
1. Generate synthetic geo-experiment data.
2. Validate parallel trends assumption.
3. Estimate treatment effect with DiD.
4. Calculate confidence intervals.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Statistics ───────────────────────────────────────────────────
import statsmodels.api as sm

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Geo-Experiment Data

In [ ]:
# Generate synthetic geo-experiment data
N_WEEKS = 20
TREATMENT_WEEK = 10  # Campaign starts at week 10

# Treatment region (campaign runs here)
weeks = list(range(N_WEEKS))
treatment_revenue = []
control_revenue = []

for week in weeks:
    # Base revenue with trend
    base = 100000 + 1000 * week
    
    # Treatment region
    if week >= TREATMENT_WEEK:
        treatment_effect = 15000  # Campaign adds $15K/week
    else:
        treatment_effect = 0
    
    treatment_revenue.append(base + treatment_effect + np.random.normal(0, 5000))
    control_revenue.append(base + np.random.normal(0, 5000))  # No treatment effect

df = pd.DataFrame({
    'week': weeks * 2,
    'region': ['Treatment'] * N_WEEKS + ['Control'] * N_WEEKS,
    'revenue': treatment_revenue + control_revenue,
    'post_treatment': [1 if w >= TREATMENT_WEEK else 0 for w in weeks] * 2
})

print(f"✓ Generated geo-experiment data")
print(f"  Weeks: {N_WEEKS}")
print(f"  Treatment starts: Week {TREATMENT_WEEK}")
print(f"  Treatment region revenue (post): ${df[(df['region']=='Treatment') & (df['post_treatment']==1)]['revenue'].mean():,.0f}")
print(f"  Control region revenue (post): ${df[(df['region']=='Control') & (df['post_treatment']==1)]['revenue'].mean():,.0f}")

---
## §4 · DiD Estimation

In [ ]:
# Prepare data for DiD regression
df['treatment'] = (df['region'] == 'Treatment').astype(int)
df['did_interaction'] = df['treatment'] * df['post_treatment']

# DiD regression: Y = β0 + β1*Treatment + β2*Post + β3*(Treatment×Post) + ε
X = df[['treatment', 'post_treatment', 'did_interaction']]
X = sm.add_constant(X)
y = df['revenue']

model = sm.OLS(y, X).fit()

print(f"✓ DiD regression complete")
print(f"\nRegression Results:")
print(f"  Intercept (β0): ${model.params['const']:,.2f}")
print(f"  Treatment (β1): ${model.params['treatment']:,.2f}")
print(f"  Post (β2): ${model.params['post_treatment']:,.2f}")
print(f"  DiD (β3): ${model.params['did_interaction']:,.2f}")
print(f"\nTreatment Effect (DiD estimate): ${model.params['did_interaction']:,.2f}")
print(f"  95% CI: [${model.conf_int().loc['did_interaction'][0]:,.2f}, ${model.conf_int().loc['did_interaction'][1]:,.2f}]")
print(f"  P-value: {model.pvalues['did_interaction']:.4f}")
print(f"  Significant: {'Yes ✓' if model.pvalues['did_interaction'] < 0.05 else 'No ✗'}")

---
## §5 · Parallel Trends Validation

In [ ]:
# Validate parallel trends assumption
pre_treatment = df[df['post_treatment'] == 0]

# Calculate pre-treatment trends
treatment_trend = pre_treatment[pre_treatment['region'] == 'Treatment']['revenue'].values
control_trend = pre_treatment[pre_treatment['region'] == 'Control']['revenue'].values

# Correlation between treatment and control pre-treatment
correlation = np.corrcoef(treatment_trend, control_trend)[0, 1]

print(f"✓ Parallel trends validation")
print(f"  Pre-treatment correlation: {correlation:.4f}")
print(f"  Interpretation: {'Strong parallel trends ✓' if correlation > 0.7 else 'Weak parallel trends ✗'}")

---
## §6 · Treatment Effect

In [ ]:
# Calculate treatment effect with bootstrap confidence intervals
n_bootstrap = 1000
bootstrap_effects = []

for _ in range(n_bootstrap):
    # Resample
    sample = df.sample(frac=1, replace=True)
    
    # Calculate DiD
    treatment_pre = sample[(sample['region']=='Treatment') & (sample['post_treatment']==0)]['revenue'].mean()
    treatment_post = sample[(sample['region']=='Treatment') & (sample['post_treatment']==1)]['revenue'].mean()
    control_pre = sample[(sample['region']=='Control') & (sample['post_treatment']==0)]['revenue'].mean()
    control_post = sample[(sample['region']=='Control') & (sample['post_treatment']==1)]['revenue'].mean()
    
    did = (treatment_post - treatment_pre) - (control_post - control_pre)
    bootstrap_effects.append(did)

# Confidence intervals
ci_lower = np.percentile(bootstrap_effects, 2.5)
ci_upper = np.percentile(bootstrap_effects, 97.5)

print(f"✓ Bootstrap analysis complete")
print(f"  Treatment effect: ${np.mean(bootstrap_effects):,.2f}")
print(f"  95% CI: [${ci_lower:,.2f}, ${ci_upper:,.2f}]")
print(f"  Weekly lift: ${np.mean(bootstrap_effects):,.2f}")
print(f"  Annual lift (projected): ${np.mean(bootstrap_effects) * 52:,.2f}")

---
## §7 · Visualization

In [ ]:
# Visualize DiD results
fig = make_subplots(rows=1, cols=2, subplot_titles=['Revenue Over Time', 'Treatment Effect Distribution'])

# Revenue over time
for region in ['Treatment', 'Control']:
    region_data = df[df['region'] == region]
    fig.add_trace(go.Scatter(
        x=region_data['week'],
        y=region_data['revenue'],
        mode='lines+markers',
        name=region
    ), row=1, col=1)

# Add treatment start line
fig.add_vline(x=TREATMENT_WEEK, line_dash="dash", line_color="red", row=1, col=1)

# Treatment effect distribution
fig.add_trace(go.Histogram(
    x=bootstrap_effects,
    nbinsx=50,
    name='Treatment Effect'
), row=1, col=2)

fig.add_vline(x=0, line_dash="dash", line_color="red", row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - Clear divergence after treatment starts")
print("   - Treatment effect is statistically significant")
print("   - Parallel trends validated pre-treatment")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Marketing Analytics
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | TV campaigns, regional tests, pricing experiments |
> | **Method** | Difference-in-Differences with parallel trends validation |
> | **Data** | Treatment and control regions, pre/post periods |
> | **Evaluation** | Statistical significance, confidence intervals |
> | **Duration** | Minimum 4-6 weeks pre/post treatment |
>
> ### 💡 Geo-Experiment Design
>
> | Step | Action | Output |
> |------|--------|--------|
> | 1 | Select matched regions | Treatment & control |
> | 2 | Validate parallel trends | Pre-treatment correlation |
> | 3 | Run campaign | Treatment region only |
> | 4 | Estimate DiD | Treatment effect |
> | 5 | Project nationally | Scale effect |
>
> ### 🔑 Key Takeaways
>
> 1. **DiD measures causal marketing impact** without user-level tracking.
> 2. **Parallel trends assumption** must be validated pre-treatment.
> 3. **Bootstrap confidence intervals** provide robust estimates.
> 4. **Geo-experiments work for TV, regional, and pricing tests**.
> 5. **Project treatment effect nationally** for budget planning.